In [ ]:
!pip install fredapi #allowing python to access to FRED data

from fredapi import Fred #importing Fred tool from fredapi
import pandas as pd #importing pandas so we can use it for short as 'pd'
import matplotlib.pyplot as plt
import numpy as np


fred = Fred(api_key= 'YOUR API KEY' ) #giving fred a variable name to be easier accessed and key so it could be used
cpi = fred.get_series("CPIAUCSL") # getting the cpi data from fred
print(cpi.head()) # .head() gives first 5 rows
print(cpi.tail()) # .tail() gives last 5 rows

In [ ]:
print(type(cpi)) #checking to see if the type was right
print(len(cpi)) #telling you the length of the rows of cpi

In [ ]:
expected = fred.get_series('MICH') #getting the expected data from fred naming it expected
print(expected.head())
print(expected.tail())
print ('starts: ', expected.index.min()) #the starting date if the data
print ('ends: ', expected.index.max()) #ending date of the data
print ('rows: ', len(expected)) # number of rows of expected inflation data
print (type(expected)) #making sure the types of the two variables are the same

In [ ]:
print('CPI: ', cpi.index.min(), 'to', cpi.index.max()) #again comparing the length of the data because survey started later
print('Expected:', expected.index.min(), 'to', expected.index.max())


In [ ]:
expected.plot(title= 'Expected Inflation% Progression', xlabel = 'Year', ylabel= 'Expected Inflation %')
#plotting the data for the expected list

In [ ]:
cpi_year = cpi.pct_change(12)*100 #comparing the percent change in cpi with the values 12 rows before
df = pd.concat([cpi_year, expected], axis = 1, keys = ['cpi_year', 'expected'])
#making a spread sheet called df with the two variables side by side
df = df.dropna() #deletes the data rows that are missing so future calculations will not be affected
print (df)

In [ ]:
cpi_year.plot(title = 'Inflation Progression')

In [ ]:
year = range (1,586,1) # starting at month 1 going up to month 585 since there is 585 months in the data set
plt.plot(year, df.iloc[:, 1], label = 'Expected', color = 'green')
plt.plot(year, df.iloc[:, 0], label = 'Actual', color = 'red')
plt.xlim(0,585)
plt.xticks((0,24,84,144,204,264,324,384,444,504,564), (1978, 1980,1985,1990,1995,2000,2005,2010,2015,2020,2025))
#making it so that the x axis ticks are in years that ends in 5 or 0 for visual, besides 1978 because that is the starting month
plt.xticks(rotation = 45) # so that the years wouldn't overlap
plt.legend(loc = 'upper right', fontsize = 10)
plt.xlabel('Year')
plt.ylabel('Percent Inflation(%)')
plt.title('Expected vs actual inflation, 1978-2026')

In [ ]:
plt.scatter(df.iloc[:,0],df.iloc[:,1])
plt.xlabel("Actual inflation (%)")
plt.ylabel("Expected inflation (%)")
plt.title("Expected vs actual inflation")
x = df.iloc[:,0] # setting x to be the list of actual inflation
y = df.iloc[:,1] # setting y to be the list of expected inflation
m, b = np.polyfit(x, y, 1) #finding the line of best fit where y= b+mx, degree 1
r = np.corrcoef(x,y)[0,1] # finding correlation coefficient
plt.plot(x, m*x + b, color = 'red', label = f'Trend: y= {m:.4f}x {b:.4f}\nCorrelation (r) = {r:.2f}')
plt.legend(loc= 'upper left')

In [ ]:
import sqlite3
conx = sqlite3.connect('inflation.db')
df.to_sql('monthly', conx, if_exists = 'replace', index_label='date') #importing to SQL as data table for later actions

In [ ]:
ten_highest_expected = pd.read_sql("SELECT date, expected FROM monthly ORDER BY expected DESC LIMIT 10",conx)
average_of_two = pd.read_sql("SELECT AVG(expected - cpi_year) AS avg_gap FROM monthly",conx)
expect_more_2_actual = pd.read_sql("SELECT * FROM monthly WHERE (expected - cpi_year) >= 2",conx)
# could replace what in the " " with your own query

In [ ]:
print(ten_highest_expected) # the 10 months with the highest expected inflation

In [ ]:
print(average_of_two) # the average of the inflation between the expected and actual

In [ ]:
print(expect_more_2_actual) # the months where the expected inflation was higher by 2 points than actual